# OVRO-LWA / EOVSA dynamic spectrum

This notebook plots the calibrated solar dynamic spectrum from the
Owens Valley Radio Observatory Long Wavelength Array (OVRO-LWA), which
covers approximately **20 - 88 MHz**.

The FITS file layout used here:

- `HDU[0].header`: includes `DATE_OBS`, `DATE_END`, image axis info
- `HDU[0].data`: spectrum of shape `(n_pol, _, n_freq, n_time)`
  (typically polarisation `0` = Stokes I)
- `HDU['SFREQ'].data['SFREQ']`: frequency axis in GHz
- `HDU['UT'].data`: pair of `mjd` (integer MJD days) and `time` (ms since
  midnight); combined to give absolute MJD per sample

Output units are SFU. The full dynamic range spans several decades, so a
logarithmic colour scale is used by default.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from matplotlib.ticker import AutoMinorLocator
from matplotlib.colors import LogNorm
import matplotlib as mpl

# Use the precise matplotlib epoch (avoids ~10 us offsets in old matplotlib).
mpl.rcParams['date.epoch'] = '1970-01-01T00:00:00'
try:
    mdates.set_epoch('1970-01-01T00:00:00')
except RuntimeError:
    pass

# Unified plotting style for all dynamic spectra notebooks.
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11

from astropy.io import fits
from astropy.time import Time
from datetime import datetime
from typing import Union, List, Tuple


## Configuration


In [ ]:
data_dir = './sample_data/ovsa'
outputs  = './outputs'
os.makedirs(outputs, exist_ok=True)

# Pick the FITS file to load.
ovsa_files = sorted(glob.glob(f'{data_dir}/*.fits'))
print(*ovsa_files, sep='\n')
filename = ovsa_files[0]


## Helper functions


In [ ]:
def subtract_background_median(df):
    """
    Subtract a per-channel median background from a dynamic spectrum.

    The function computes the median along the time axis (axis=0) for each
    frequency channel, then subtracts it from every time sample of that
    channel. This is the standard approach for highlighting transient
    emission against a slowly-varying instrumental/sky background.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame with shape (n_times, n_freqs). Index is time, columns
        are frequencies.

    Returns
    -------
    pandas.DataFrame
        Same shape as input with the per-channel median removed.
    """
    bkg = np.nanmedian(df.values, axis=0)
    return df - np.tile(bkg, (df.shape[0], 1))


def find_nearest_datetime(
    times: Union[List, np.ndarray],
    target: Union[str, datetime],
) -> Tuple[int, datetime]:
    """
    Return the index and value of the datetime nearest to `target`.

    Parameters
    ----------
    times : list or numpy.ndarray
        Sequence of datetime objects or ISO-format strings.
    target : str or datetime
        Target datetime (ISO string or datetime).

    Returns
    -------
    (int, datetime)
        Index of the nearest entry and its value as a datetime.
    """
    target_dt = datetime.fromisoformat(target) if isinstance(target, str) else target
    times_dt = ([datetime.fromisoformat(t) for t in times]
                if isinstance(times[0], str) else list(times))
    differences = [abs((t - target_dt).total_seconds()) for t in times_dt]
    idx = int(np.argmin(differences))
    return idx, times_dt[idx]


## Load the FITS file


In [ ]:
hdul = fits.open(filename)
hdul.info()

# Primary data cube: select Stokes I (polarisation index 0).
spec_pol = np.asarray(hdul[0].data)[:, 0, :, :][0]
print(f'Spectrum shape (freq, time): {spec_pol.shape}')

# Frequency axis (GHz in the file; we convert to MHz at plot time).
freq_ghz = np.asarray(hdul['SFREQ'].data['SFREQ'])

# Time axis: MJD (days) + ms-of-day -> Time -> python datetime.
ut = hdul['UT'].data
mjd  = ut['mjd']
ms   = ut['time']
time_mjd = mjd + ms / 86400000.0
time_dt = Time(time_mjd, format='mjd').to_datetime()

# Observation start/end from the header.
st = hdul[0].header['DATE_OBS'].replace('T', ' ').split('.')[0]
et = hdul[0].header['DATE_END'].replace('T', ' ').split('.')[0]
print(f'Observation window: {st} - {et}')


## Remove the per-channel median background


In [ ]:
# Build a DataFrame for clean per-channel background subtraction.
df_ovsa = pd.DataFrame(spec_pol.T, index=time_dt, columns=freq_ghz)
df_ovsa_nobkg = subtract_background_median(df_ovsa)


## Plot the full dynamic spectrum


In [ ]:
# OVRO-LWA spans several decades in flux density; use a log colour scale.
positive = df_ovsa.values[df_ovsa.values > 0]
norm = LogNorm(
    vmin=np.nanpercentile(positive, 5),
    vmax=np.nanpercentile(positive, 99),
)

fig = plt.figure(figsize=(12, 5))
ax  = fig.add_subplot(111)
pc  = ax.pcolormesh(
    df_ovsa.index, df_ovsa.columns * 1e3, df_ovsa.values.T,
    norm=norm, cmap='Spectral_r',
)
fig.colorbar(pc, ax=ax, pad=0.02, label='Intensity (SFU)')

ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.set_title(f'OVRO-LWA dynamic spectrum  -  {st} - {et}')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))

fig.tight_layout()
fig.savefig(f'{outputs}/ovsa_dyspec_{time_dt[0].date()}.png', bbox_inches='tight')
plt.show()


## Optional: zoom into a sub-window


In [ ]:
# Choose a sub-window for closer inspection.
st_target = f'{time_dt[0].date()}T{time_dt[0].strftime("%H:%M:%S")}'
et_target = f'{time_dt[-1].date()}T{time_dt[-1].strftime("%H:%M:%S")}'

# Replace the lines below with concrete timestamps if needed, e.g.
# st_target = '2026-01-18T17:42:00'
# et_target = '2026-01-18T18:15:00'

st_idx, _ = find_nearest_datetime(time_dt, st_target)
et_idx, _ = find_nearest_datetime(time_dt, et_target)

sub_spec = spec_pol[:, st_idx:et_idx]
sub_time = time_dt[st_idx:et_idx]

positive = sub_spec[sub_spec > 0]
norm = LogNorm(
    vmin=np.nanpercentile(positive, 5),
    vmax=np.nanpercentile(positive, 99.9),
)

fig = plt.figure(figsize=(12, 5))
ax  = fig.add_subplot(111)
pc  = ax.pcolormesh(sub_time, freq_ghz * 1e3, sub_spec,
                    norm=norm, cmap='Spectral_r')
fig.colorbar(pc, ax=ax, pad=0.02, label='Intensity (SFU)')

ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.set_title(f'OVRO-LWA dynamic spectrum  -  {sub_time[0]} - {sub_time[-1]}')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))

fig.tight_layout()
plt.show()
